# CS5487 Digits Project In Colab

> Default cloud-workspace layout: when this notebook runs inside a Tencent Cloud or similar workspace mounted at `/workspace`, keep `PROJECT_ROOT = "/workspace"` and store artifacts in the repo-local `artifacts/` directory unless you explicitly need a separate storage root.

Use this notebook as the primary execution surface when local CPU and memory are limited. It can pull the project source from GitHub at startup, keep `artifacts/` persistent in Google Drive, or keep them in a workspace-local folder that you can archive before the session resets.

In [ ]:
USE_GOOGLE_DRIVE = False
SYNC_PROJECT_FROM_GITHUB = False
GITHUB_REPO_URL = "https://github.com/MJWade96/CS5487-Project"
GITHUB_REF = None
WORKSPACE_ROOT = "/workspace"
PROJECT_ROOT = (
    f"{WORKSPACE_ROOT}/CS5487-Project"
    if SYNC_PROJECT_FROM_GITHUB
    else WORKSPACE_ROOT
)
# Keep artifacts inside the active workspace checkout by default so cloud notebooks and source stay together.
# Set ARTIFACTS_ROOT = RUNTIME_ARTIFACTS_ROOT if you want a separate writable storage root.
RUNTIME_ARTIFACTS_ROOT = f"{WORKSPACE_ROOT}/runtime-storage"
ARTIFACTS_ROOT = (
    "/content/drive/MyDrive/CS5487 Course Project Code"
    if USE_GOOGLE_DRIVE
    else None
)
# Export a zip only when you intentionally keep artifacts outside the repo tree.
EXPORT_RUNTIME_ARTIFACTS_ARCHIVE = False
# Browser-triggered downloads are only available in Colab.
DOWNLOAD_RUNTIME_ARTIFACTS_ARCHIVE = False
GRID_SEARCH_JOBS = 2
# Use one of: None, "light", "heavy", "rbf_only", "random_forest_only", "mlp_only".
# When set, BATCH_PRESET overrides RUN_NAME and SELECTED_MODEL_NAMES.
BATCH_PRESET = "heavy"
# Set False when you only want to inspect or combine finished batch runs.
RUN_EXPERIMENTS = True
# Set True to rebuild canonical outputs from COMBINE_RUN_NAMES after a fresh run finishes.
COMBINE_AFTER_RUN = False
SELECTED_TRIAL_NAMES = None
SELECTED_MODEL_NAMES = ["knn_1", "logistic_regression_ova", "linear_svm_ova"]
RUN_NAME = "batch_light_v2"
# Use the actual completed run folder names here, for example ["batch_light_v2", "batch_heavy"].
COMBINE_RUN_NAMES = ["batch_light_v2", "batch_heavy"]

In [ ]:
from pathlib import Path
import importlib
import shutil
import subprocess


def _run_git(project_root: Path, *args: str, capture_output: bool = False) -> str | None:
    command = ["git", "-C", str(project_root), *args]
    if capture_output:
        return subprocess.check_output(command, text=True).strip()
    subprocess.check_call(command)
    return None


def _resolve_git_sync_target(project_root: Path, requested_ref: str | None) -> tuple[str, bool]:
    if requested_ref is not None:
        remote_branch_matches = _run_git(
            project_root,
            "ls-remote",
            "--heads",
            "origin",
            requested_ref,
            capture_output=True,
        )
        return requested_ref, bool(remote_branch_matches)

    remote_head = _run_git(
        project_root,
        "symbolic-ref",
        "--short",
        "refs/remotes/origin/HEAD",
        capture_output=True,
    )
    return remote_head.split("/", 1)[1], True


def _sync_git_checkout(project_root: Path, repo_url: str, requested_ref: str | None) -> None:
    # Route every sync through one helper so notebook startup does not depend on local branch tracking state.
    _run_git(project_root, "remote", "set-url", "origin", repo_url)
    _run_git(project_root, "fetch", "origin", "--tags")
    _run_git(project_root, "remote", "set-head", "origin", "--auto")

    target_ref, is_remote_branch = _resolve_git_sync_target(project_root, requested_ref)
    if is_remote_branch:
        _run_git(project_root, "checkout", "-B", target_ref, f"origin/{target_ref}")
        _run_git(project_root, "branch", "--set-upstream-to", f"origin/{target_ref}", target_ref)
    else:
        _run_git(project_root, "checkout", target_ref)


if USE_GOOGLE_DRIVE:
    colab_drive = importlib.import_module("google.colab.drive")
    colab_drive.mount("/content/drive")

project_root = Path(PROJECT_ROOT).expanduser()
artifacts_root = None if ARTIFACTS_ROOT is None else Path(ARTIFACTS_ROOT).expanduser()

if SYNC_PROJECT_FROM_GITHUB:
    project_root.parent.mkdir(parents=True, exist_ok=True)
    git_dir = project_root / ".git"
    if git_dir.exists():
        _sync_git_checkout(project_root, GITHUB_REPO_URL, GITHUB_REF)
    elif project_root.exists():
        raise FileExistsError(
            "PROJECT_ROOT already exists but is not a git checkout. "
            "Choose a clean folder, or remove the old manual copy first."
        )
    else:
        subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(project_root)])
        if GITHUB_REF is not None:
            _sync_git_checkout(project_root, GITHUB_REPO_URL, GITHUB_REF)
elif not project_root.exists():
    raise FileNotFoundError(
        "Project root not found. Enable SYNC_PROJECT_FROM_GITHUB or update PROJECT_ROOT."
    )

required_children = [
    project_root / "requirements.txt",
    project_root / "src",
    project_root / "digits4000_txt",
    project_root / "challenge",
]
missing_children = [str(path.relative_to(project_root)) for path in required_children if not path.exists()]
if missing_children:
    raise FileNotFoundError(
        "Project root is missing required paths: " + ", ".join(missing_children)
    )

if artifacts_root is not None:
    artifacts_root.mkdir(parents=True, exist_ok=True)
    external_artifacts_dir = artifacts_root / "artifacts"
    repo_artifacts_dir = project_root / "artifacts"

    if repo_artifacts_dir.is_symlink():
        if repo_artifacts_dir.resolve() != external_artifacts_dir.resolve():
            repo_artifacts_dir.unlink()
            repo_artifacts_dir.symlink_to(external_artifacts_dir, target_is_directory=True)
    else:
        if repo_artifacts_dir.exists() and not external_artifacts_dir.exists():
            shutil.copytree(repo_artifacts_dir, external_artifacts_dir)
        elif not external_artifacts_dir.exists():
            external_artifacts_dir.mkdir(parents=True, exist_ok=True)

        if repo_artifacts_dir.exists():
            shutil.rmtree(repo_artifacts_dir)
        repo_artifacts_dir.symlink_to(external_artifacts_dir, target_is_directory=True)

# Keep optional external artifacts outside the repo checkout so they can be archived separately.
def export_runtime_artifacts_archive(project_root, artifacts_root, run_name=None, download=False):
    if USE_GOOGLE_DRIVE:
        print("Runtime archive export is disabled because Google Drive persistence is enabled.")
        return None

    source_dir = project_root / "artifacts" if artifacts_root is None else artifacts_root / "artifacts"
    if not source_dir.exists():
        print("No artifacts directory exists yet, so there is nothing to archive.")
        return None

    exports_parent = project_root if project_root.parent == Path(project_root.anchor) else project_root.parent
    exports_dir = exports_parent / "exports"
    exports_dir.mkdir(parents=True, exist_ok=True)
    archive_stem = "artifacts_snapshot" if run_name is None else f"artifacts_snapshot_{run_name}"
    archive_path = Path(
        shutil.make_archive(
            str(exports_dir / archive_stem),
            "zip",
            root_dir=source_dir.parent,
            base_dir=source_dir.name,
        )
    )
    print("Runtime artifacts archive:", archive_path)
    print("Download this archive before the workspace session resets.")

    if download:
        try:
            colab_files = importlib.import_module("google.colab.files")
        except ModuleNotFoundError:
            print("Browser download is only available inside Colab.")
        else:
            colab_files.download(str(archive_path))

    return archive_path

git_head = subprocess.check_output(
    ["git", "-C", str(project_root), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
actual_artifacts_dir = project_root / "artifacts" if artifacts_root is None else artifacts_root / "artifacts"
print("Project root:", project_root)
print("Artifacts directory:", actual_artifacts_dir)
print("Git revision:", git_head)
if USE_GOOGLE_DRIVE:
    print("Artifact mode: persistent Google Drive storage.")
elif artifacts_root is None:
    print("Artifact mode: repository-local storage inside the workspace checkout.")
else:
    print("Artifact mode: external workspace storage.")
    print("These files survive cell reruns in the current session and should be archived if the runtime is ephemeral.")

In [ ]:
import importlib
import os
import subprocess
import sys

os.chdir(project_root)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Force Python to reload the freshly synced checkout instead of reusing stale modules.
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == 'digits_project' or module_name.startswith('digits_project.'):
        del sys.modules[module_name]

print('Working directory:', project_root)
print('Cleared cached digits_project modules after source sync.')

In [ ]:
import inspect
import json
from pathlib import Path

import pandas as pd

import digits_project.config as project_config
import digits_project.experiment as experiment_module

configure_project = project_config.configure_project
run_project_experiments = getattr(experiment_module, "run_project_experiments", None)
combine_experiment_runs = getattr(experiment_module, "combine_experiment_runs", None)

if run_project_experiments is None:
    raise ImportError(
        "digits_project.experiment is missing run_project_experiments. "
        "Refresh the GitHub checkout in Cell 4 and rerun from Cell 4."
    )

configure_project_signature = inspect.signature(configure_project)
run_project_experiments_signature = inspect.signature(run_project_experiments)


def _call_configure_project(root_dir, grid_search_jobs, run_name):
    kwargs = {
        "root_dir": root_dir,
        "grid_search_jobs": grid_search_jobs,
    }
    if "run_name" in configure_project_signature.parameters:
        kwargs["run_name"] = run_name
    elif run_name is not None:
        print(
            "configure_project in this source tree does not accept run_name; "
            "outputs will use the default artifacts directory."
        )
    return configure_project(**kwargs)


def _call_run_project_experiments(
    root_dir,
    grid_search_jobs,
    selected_trial_names,
    selected_model_names,
    run_name,
):
    kwargs = {
        "root_dir": root_dir,
        "grid_search_jobs": grid_search_jobs,
        "selected_trial_names": selected_trial_names,
        "selected_model_names": selected_model_names,
    }
    if "run_name" in run_project_experiments_signature.parameters:
        kwargs["run_name"] = run_name
    elif run_name is not None:
        print(
            "run_project_experiments in this source tree does not accept run_name; "
            "batch outputs will use the default artifacts directory."
        )
    return run_project_experiments(**kwargs)


def _build_summary_frames(final_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_frame = (
        final_frame.groupby("model", as_index=False)
        .agg(
            mnist_accuracy_mean=("mnist_accuracy", "mean"),
            mnist_accuracy_std=("mnist_accuracy", "std"),
            challenge_accuracy_mean=("challenge_accuracy", "mean"),
            challenge_accuracy_std=("challenge_accuracy", "std"),
            mnist_macro_f1_mean=("mnist_macro_f1", "mean"),
            challenge_macro_f1_mean=("challenge_macro_f1", "mean"),
        )
        .sort_values("mnist_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )

    email_frame = final_frame[["trial", "model", "mnist_accuracy", "challenge_accuracy"]].copy()
    email_frame["challenge_reference_knn_mean"] = project_config.CHALLENGE_REFERENCE_ACCURACY
    return summary_frame, email_frame


def _safe_mode(values: pd.Series):
    modes = values.mode()
    return None if modes.empty else modes.iloc[0]


def _consistency_ratio(values: pd.Series) -> float:
    if values.empty:
        return 0.0
    return float(values.value_counts(normalize=True).iloc[0])


def _build_model_tradeoff_frame(final_frame: pd.DataFrame) -> pd.DataFrame:
    if final_frame.empty:
        return pd.DataFrame(
            columns=[
                "model",
                "trial_count",
                "selected_preprocessing_mode",
                "selected_preprocessing_unique_count",
                "preprocessing_consistency_ratio",
                "mnist_accuracy_mean",
                "mnist_accuracy_std",
                "mnist_accuracy_range",
                "mnist_macro_f1_mean",
                "challenge_accuracy_mean",
                "challenge_accuracy_std",
                "challenge_accuracy_range",
                "challenge_macro_f1_mean",
                "model_selection_runtime_mean_seconds",
                "model_selection_runtime_std_seconds",
            ]
        )

    return (
        final_frame.groupby("model", as_index=False)
        .agg(
            trial_count=("trial", "count"),
            selected_preprocessing_mode=("selected_preprocessing", _safe_mode),
            selected_preprocessing_unique_count=("selected_preprocessing", "nunique"),
            preprocessing_consistency_ratio=("selected_preprocessing", _consistency_ratio),
            mnist_accuracy_mean=("mnist_accuracy", "mean"),
            mnist_accuracy_std=("mnist_accuracy", "std"),
            mnist_accuracy_range=("mnist_accuracy", lambda values: float(values.max() - values.min())),
            mnist_macro_f1_mean=("mnist_macro_f1", "mean"),
            challenge_accuracy_mean=("challenge_accuracy", "mean"),
            challenge_accuracy_std=("challenge_accuracy", "std"),
            challenge_accuracy_range=("challenge_accuracy", lambda values: float(values.max() - values.min())),
            challenge_macro_f1_mean=("challenge_macro_f1", "mean"),
            model_selection_runtime_mean_seconds=("model_selection_runtime_seconds", "mean"),
            model_selection_runtime_std_seconds=("model_selection_runtime_seconds", "std"),
        )
        .sort_values("mnist_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )


def _build_preprocessing_tradeoff_frame(cv_frame: pd.DataFrame, final_frame: pd.DataFrame) -> pd.DataFrame:
    if cv_frame.empty:
        return pd.DataFrame(
            columns=[
                "model",
                "preprocessing",
                "trial_count",
                "best_cv_accuracy_mean",
                "best_cv_accuracy_std",
                "runtime_seconds_mean",
                "runtime_seconds_std",
                "selected_trial_count",
                "selected_trial_ratio",
            ]
        )

    tradeoff_frame = (
        cv_frame.groupby(["model", "preprocessing"], as_index=False)
        .agg(
            trial_count=("trial", "count"),
            best_cv_accuracy_mean=("best_cv_accuracy", "mean"),
            best_cv_accuracy_std=("best_cv_accuracy", "std"),
            runtime_seconds_mean=("runtime_seconds", "mean"),
            runtime_seconds_std=("runtime_seconds", "std"),
        )
    )

    if final_frame.empty:
        selected_counts = pd.DataFrame(columns=["model", "preprocessing", "selected_trial_count"])
    else:
        selected_counts = (
            final_frame.rename(columns={"selected_preprocessing": "preprocessing"})
            .groupby(["model", "preprocessing"], as_index=False)
            .agg(selected_trial_count=("trial", "count"))
        )

    tradeoff_frame = tradeoff_frame.merge(selected_counts, on=["model", "preprocessing"], how="left")
    tradeoff_frame["selected_trial_count"] = tradeoff_frame["selected_trial_count"].fillna(0).astype(int)
    tradeoff_frame["selected_trial_ratio"] = tradeoff_frame["selected_trial_count"] / tradeoff_frame["trial_count"]
    return tradeoff_frame.sort_values(["model", "best_cv_accuracy_mean"], ascending=[True, False]).reset_index(drop=True)


def _merge_result_frames(frames: list[pd.DataFrame], key_columns: list[str]) -> pd.DataFrame:
    non_empty_frames = [frame for frame in frames if not frame.empty]
    if not non_empty_frames:
        raise ValueError("No result rows were available to combine.")
    merged = pd.concat(non_empty_frames, ignore_index=True, sort=False)
    return merged.drop_duplicates(subset=key_columns, keep="last")


def _resolve_batch_run_names(root_dir: Path, run_names) -> list[str]:
    if run_names is not None:
        if isinstance(run_names, str):
            return [run_names]
        return sorted(set(run_names))

    runs_dir = project_config.ProjectPaths(root_dir).runs_dir
    if not runs_dir.exists():
        raise FileNotFoundError(f"No batch run directory found under {runs_dir}")

    available_names = sorted(path.name for path in runs_dir.iterdir() if path.is_dir())
    if not available_names:
        raise ValueError("No batch runs are available to combine.")
    return available_names


def _save_accuracy_comparison_plot(summary_frame: pd.DataFrame, path: Path) -> None:
    import matplotlib.pyplot as plt

    try:
        import seaborn as sns
    except ImportError:
        sns = None

    plt.figure(figsize=(10, 5))
    if summary_frame.empty:
        plt.text(0.5, 0.5, "No summary rows", ha="center", va="center")
        plt.axis("off")
    elif sns is None:
        plt.bar(summary_frame["model"], summary_frame["mnist_accuracy_mean"], color="#2c7fb8")
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Mean accuracy on official test set")
        plt.xlabel("Model")
    else:
        sns.barplot(data=summary_frame, x="model", y="mnist_accuracy_mean", color="#2c7fb8")
        plt.ylabel("Mean accuracy on official test set")
        plt.xlabel("Model")
        plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def _save_accuracy_runtime_tradeoff_plot(tradeoff_frame: pd.DataFrame, path: Path) -> None:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(9, 5.5))
    if tradeoff_frame.empty:
        plt.text(0.5, 0.5, "No trade-off rows", ha="center", va="center")
        plt.axis("off")
    else:
        x_values = tradeoff_frame["model_selection_runtime_mean_seconds"]
        y_values = tradeoff_frame["mnist_accuracy_mean"]
        x_errors = tradeoff_frame["model_selection_runtime_std_seconds"].fillna(0.0)
        y_errors = tradeoff_frame["mnist_accuracy_std"].fillna(0.0)
        plt.errorbar(
            x_values,
            y_values,
            xerr=x_errors,
            yerr=y_errors,
            fmt="o",
            ecolor="#8c96a0",
            color="#2c7fb8",
            capsize=4,
        )
        for row in tradeoff_frame.itertuples(index=False):
            plt.annotate(
                row.model,
                (row.model_selection_runtime_mean_seconds, row.mnist_accuracy_mean),
                textcoords="offset points",
                xytext=(6, 4),
                fontsize=9,
            )
        plt.xlabel("Mean model-selection runtime (seconds)")
        plt.ylabel("Mean official-test accuracy")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def _save_result_tables(
    cv_frame: pd.DataFrame,
    cv_detailed_frame: pd.DataFrame,
    final_frame: pd.DataFrame,
    summary_frame: pd.DataFrame,
    email_frame: pd.DataFrame,
    paths,
    source_runs=None,
    extra_metadata=None,
):
    case_examples_dir = getattr(paths, "case_examples_dir", paths.results_dir / "case_examples")
    for path in (
        paths.results_dir,
        paths.figures_dir,
        paths.models_dir,
        paths.predictions_dir,
        paths.per_class_dir,
        case_examples_dir,
    ):
        path.mkdir(parents=True, exist_ok=True)

    model_tradeoff_frame = _build_model_tradeoff_frame(final_frame)
    preprocessing_tradeoff_frame = _build_preprocessing_tradeoff_frame(cv_frame, final_frame)

    cv_frame.to_csv(paths.results_dir / "cv_leaderboard.csv", index=False)
    final_frame.to_csv(paths.results_dir / "final_selected_models.csv", index=False)
    summary_frame.to_csv(paths.results_dir / "summary_by_model.csv", index=False)
    email_frame.to_csv(paths.results_dir / "email_summary.csv", index=False)
    model_tradeoff_frame.to_csv(paths.results_dir / "model_tradeoff_summary.csv", index=False)
    preprocessing_tradeoff_frame.to_csv(paths.results_dir / "preprocessing_tradeoff_summary.csv", index=False)

    detailed_path = paths.results_dir / "cv_results_detailed.csv"
    if cv_detailed_frame is not None and not cv_detailed_frame.empty:
        cv_detailed_frame.to_csv(detailed_path, index=False)
    elif detailed_path.exists():
        detailed_path.unlink()

    _save_accuracy_comparison_plot(summary_frame, paths.figures_dir / "mnist_accuracy_by_model.png")
    _save_accuracy_runtime_tradeoff_plot(model_tradeoff_frame, paths.figures_dir / "mnist_accuracy_vs_runtime.png")

    protocol_payload = {
        "challenge_reference_accuracy": project_config.CHALLENGE_REFERENCE_ACCURACY,
        "public_material_can_include_challenge": False,
    }
    if source_runs is not None:
        protocol_payload["source_runs"] = list(source_runs)
    if extra_metadata:
        protocol_payload["artifact_metadata"] = extra_metadata
    (paths.results_dir / "challenge_protocol.json").write_text(
        json.dumps(protocol_payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return model_tradeoff_frame, preprocessing_tradeoff_frame


if combine_experiment_runs is None:
    print(
        "combine_experiment_runs is missing in this source tree; "
        "the notebook fallback will be used for summary and trade-off rebuilding."
    )

    def combine_experiment_runs(root_dir=None, run_names=None) -> dict[str, pd.DataFrame]:
        resolved_root = (
            project_config.ROOT_DIR
            if root_dir is None
            else Path(root_dir).expanduser().resolve()
        )
        selected_run_names = _resolve_batch_run_names(resolved_root, run_names)
        combined_paths = project_config.ProjectPaths(resolved_root)

        cv_frames = []
        final_frames = []
        cv_detailed_frames = []
        missing_detailed_cv_runs = []
        for run_name in selected_run_names:
            batch_paths = project_config.ProjectPaths(resolved_root, run_name=run_name)
            cv_path = batch_paths.results_dir / "cv_leaderboard.csv"
            final_path = batch_paths.results_dir / "final_selected_models.csv"
            if not cv_path.exists():
                raise FileNotFoundError(f"Missing required results file: {cv_path}")
            if not final_path.exists():
                raise FileNotFoundError(f"Missing required results file: {final_path}")
            cv_frames.append(pd.read_csv(cv_path))
            final_frames.append(pd.read_csv(final_path))

            detailed_path = batch_paths.results_dir / "cv_results_detailed.csv"
            if detailed_path.exists():
                cv_detailed_frames.append(pd.read_csv(detailed_path))
            else:
                missing_detailed_cv_runs.append(run_name)

        cv_frame = _merge_result_frames(cv_frames, ["trial", "model", "preprocessing"])
        cv_frame = cv_frame.sort_values(
            ["trial", "model", "best_cv_accuracy"],
            ascending=[True, True, False],
        ).reset_index(drop=True)

        final_frame = _merge_result_frames(final_frames, ["trial", "model"])
        final_frame = final_frame.sort_values(["model", "trial"]).reset_index(drop=True)
        cv_detailed_frame = (
            _merge_result_frames(cv_detailed_frames, ["trial", "model", "preprocessing", "params_json"])
            if cv_detailed_frames
            else pd.DataFrame(columns=["trial", "model", "preprocessing", "params_json"])
        )
        summary_frame, email_frame = _build_summary_frames(final_frame)
        extra_metadata = None
        if missing_detailed_cv_runs:
            extra_metadata = {"missing_detailed_cv_runs": missing_detailed_cv_runs}
        model_tradeoff_frame, preprocessing_tradeoff_frame = _save_result_tables(
            cv_frame,
            cv_detailed_frame,
            final_frame,
            summary_frame,
            email_frame,
            combined_paths,
            source_runs=selected_run_names,
            extra_metadata=extra_metadata,
        )
        print(
            "Notebook fallback rebuilt the summary/trade-off tables. "
            "Sync the updated source tree if you also need canonical predictions, per-class files, models, and case examples rebuilt."
        )

        return {
            "cv": cv_frame,
            "cv_detailed": cv_detailed_frame,
            "final": final_frame,
            "summary": summary_frame,
            "email": email_frame,
            "model_tradeoff": model_tradeoff_frame,
            "preprocessing_tradeoff": preprocessing_tradeoff_frame,
        }


BATCH_PRESETS = {
    "light": {
        "run_name": "batch_light",
        "selected_model_names": [
            "knn_1",
            "logistic_regression_ova",
            "linear_svm_ova",
        ],
    },
    "heavy": {
        "run_name": "batch_heavy",
        "selected_model_names": [
            "rbf_svm_ova",
            "random_forest",
            "mlp",
        ],
    },
    "rbf_only": {
        "run_name": "batch_rbf_svm",
        "selected_model_names": ["rbf_svm_ova"],
    },
    "random_forest_only": {
        "run_name": "batch_random_forest",
        "selected_model_names": ["random_forest"],
    },
    "mlp_only": {
        "run_name": "batch_mlp",
        "selected_model_names": ["mlp"],
    },
}

if BATCH_PRESET is not None:
    if BATCH_PRESET not in BATCH_PRESETS:
        raise ValueError(f"Unknown BATCH_PRESET: {BATCH_PRESET}")
    RUN_NAME = BATCH_PRESETS[BATCH_PRESET]["run_name"]
    SELECTED_MODEL_NAMES = BATCH_PRESETS[BATCH_PRESET]["selected_model_names"]

_call_configure_project(root_dir=project_root, grid_search_jobs=GRID_SEARCH_JOBS, run_name=RUN_NAME)
print(project_config.get_runtime_config())
print("Selected trials:", SELECTED_TRIAL_NAMES)
print("Selected models:", SELECTED_MODEL_NAMES)
print("Combine run names:", COMBINE_RUN_NAMES)

In [ ]:
from digits_project.data import load_digits_project_data

bundle = load_digits_project_data()
print('digits4000 shape:', bundle.X.shape)
print('digits4000 labels:', bundle.y.shape)
print('challenge shape:', bundle.challenge_X.shape)
print('challenge labels:', bundle.challenge_y.shape)
print('trials:', [trial.name for trial in bundle.trials])

In [ ]:
paths = project_config.PATHS
results_dir = paths.results_dir
predictions_dir = paths.predictions_dir
per_class_dir = paths.per_class_dir
case_examples_dir = getattr(paths, "case_examples_dir", results_dir / "case_examples")
runs_dir = paths.runs_dir

for directory in [results_dir, predictions_dir, per_class_dir, case_examples_dir, paths.models_dir, runs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

cv_path = results_dir / "cv_leaderboard.csv"
final_path = results_dir / "final_selected_models.csv"
detailed_cv_path = results_dir / "cv_results_detailed.csv"
model_tradeoff_path = results_dir / "model_tradeoff_summary.csv"
preprocessing_tradeoff_path = results_dir / "preprocessing_tradeoff_summary.csv"
protocol_path = results_dir / "challenge_protocol.json"

required_outputs = {
    "cv leaderboard": cv_path,
    "final selected models": final_path,
    "detailed CV results": detailed_cv_path,
    "model trade-off summary": model_tradeoff_path,
    "preprocessing trade-off summary": preprocessing_tradeoff_path,
    "challenge protocol": protocol_path,
}

missing_outputs = [name for name, path in required_outputs.items() if not path.exists()]
if missing_outputs:
    print("Missing canonical outputs:")
    for name in missing_outputs:
        print(f" - {name}")
else:
    print("All canonical summary outputs are present.")

if cv_path.exists():
    cv_frame = pd.read_csv(cv_path)
    completed_cv = cv_frame[["trial", "model", "preprocessing"]].drop_duplicates()
    print(f"Completed CV rows: {len(completed_cv)}")
    display(completed_cv.sort_values(["trial", "model", "preprocessing"]).reset_index(drop=True))
else:
    cv_frame = pd.DataFrame(columns=["trial", "model", "preprocessing"])
    completed_cv = cv_frame.copy()

if final_path.exists():
    final_frame = pd.read_csv(final_path)
    completed_models = final_frame[["trial", "model", "selected_preprocessing"]].copy()
    print(f"Completed final evaluations: {len(completed_models)}")
    display(completed_models.sort_values(["trial", "model"]).reset_index(drop=True))
else:
    final_frame = pd.DataFrame(columns=["trial", "model", "selected_preprocessing"])
    completed_models = final_frame.copy()

if detailed_cv_path.exists():
    detailed_cv_frame = pd.read_csv(detailed_cv_path)
    print(f"Detailed CV rows available: {len(detailed_cv_frame)}")
else:
    detailed_cv_frame = pd.DataFrame()
    print("Detailed CV rows available: 0")

# Reuse the notebook's selection settings while handling the "None means all" convention.
if SELECTED_TRIAL_NAMES is None:
    audit_bundle = globals().get("bundle")
    if audit_bundle is None:
        from digits_project.data import load_digits_project_data

        audit_bundle = load_digits_project_data()
    selected_trial_names = [trial.name for trial in audit_bundle.trials]
else:
    selected_trial_names = list(SELECTED_TRIAL_NAMES)

if SELECTED_MODEL_NAMES is None:
    selected_model_names = [model_spec.name for model_spec in experiment_module.MODEL_SPECS]
else:
    selected_model_names = list(SELECTED_MODEL_NAMES)

pending_pairs = []
for trial_name in selected_trial_names:
    for model_name in selected_model_names:
        if not (
            (completed_models["trial"] == trial_name) &
            (completed_models["model"] == model_name)
        ).any():
            pending_pairs.append((trial_name, model_name))

print(f"Pending trial/model pairs: {len(pending_pairs)}")
for trial_name, model_name in pending_pairs:
    print(f" - {trial_name} / {model_name}")

if COMBINE_RUN_NAMES is None:
    combine_targets = sorted(path.name for path in runs_dir.iterdir() if path.is_dir())
else:
    combine_targets = list(COMBINE_RUN_NAMES)

if combine_targets:
    print("Selected batch runs for combine/audit:", combine_targets)
    missing_run_details = []
    for run_name in combine_targets:
        batch_paths = project_config.ProjectPaths(project_root, run_name=run_name)
        batch_missing = [
            label
            for label, path in {
                "cv_leaderboard": batch_paths.results_dir / "cv_leaderboard.csv",
                "final_selected_models": batch_paths.results_dir / "final_selected_models.csv",
                "cv_results_detailed": batch_paths.results_dir / "cv_results_detailed.csv",
            }.items()
            if not path.exists()
        ]
        if batch_missing:
            missing_run_details.append((run_name, batch_missing))

    if missing_run_details:
        print("Batch runs with missing files:")
        for run_name, missing_files in missing_run_details:
            print(f" - {run_name}: {', '.join(missing_files)}")
    else:
        print("All selected batch runs contain the expected summary files.")
else:
    print("No batch runs selected for combine/audit.")

prediction_files = sorted(predictions_dir.glob("*.csv"))
per_class_files = sorted(per_class_dir.glob("*.csv"))
case_example_files = sorted(case_examples_dir.glob("*.csv"))
figure_files = sorted(paths.figures_dir.glob("*.png"))
model_files = sorted(paths.models_dir.glob("*.joblib"))

print(f"Prediction files: {len(prediction_files)}")
print(f"Per-class metric files: {len(per_class_files)}")
print(f"Case example files: {len(case_example_files)}")
print(f"Figure files: {len(figure_files)}")
print(f"Saved model files: {len(model_files)}")

artifact_warnings = []
if cv_path.exists() and not prediction_files:
    artifact_warnings.append("canonical prediction CSVs are missing")
if cv_path.exists() and not per_class_files:
    artifact_warnings.append("canonical per-class CSVs are missing")
if final_path.exists() and not model_files:
    artifact_warnings.append("selected model artifacts are missing")
if final_path.exists() and not case_example_files:
    artifact_warnings.append("representative case-example CSVs are missing")
if final_path.exists() and not figure_files:
    artifact_warnings.append("canonical figures are missing")
if final_path.exists() and not detailed_cv_path.exists():
    artifact_warnings.append("parameter-sensitivity analysis is unavailable until cv_results_detailed.csv is generated")

protocol_payload = None
if protocol_path.exists():
    protocol_payload = json.loads(protocol_path.read_text(encoding="utf-8"))
    missing_detailed_runs = protocol_payload.get("artifact_metadata", {}).get("missing_detailed_cv_runs", [])
    if missing_detailed_runs:
        artifact_warnings.append(
            "combine inputs missing detailed CV rows for: " + ", ".join(missing_detailed_runs)
        )

if artifact_warnings:
    print("Artifact warnings:")
    for warning in artifact_warnings:
        print(f" - {warning}")
else:
    print("Artifact audit passed for the currently selected scope.")

if model_tradeoff_path.exists():
    print("Model trade-off summary preview:")
    display(pd.read_csv(model_tradeoff_path).head())

if preprocessing_tradeoff_path.exists():
    print("Preprocessing trade-off summary preview:")
    display(pd.read_csv(preprocessing_tradeoff_path).head())

if case_example_files:
    print("Sample case-example rows:")
    display(pd.read_csv(case_example_files[0]).head())

In [ ]:
results = None
if RUN_EXPERIMENTS:
    results = _call_run_project_experiments(
        root_dir=project_root,
        grid_search_jobs=GRID_SEARCH_JOBS,
        selected_trial_names=SELECTED_TRIAL_NAMES,
        selected_model_names=SELECTED_MODEL_NAMES,
        run_name=RUN_NAME,
    )
else:
    print("RUN_EXPERIMENTS is False, so this notebook will not start a new training run.")

In [ ]:
combined_results = None
if RUN_EXPERIMENTS:
    # Reuse Cell 8 output so the notebook does not launch the same training run twice.
    if results is None:
        raise RuntimeError(
            "No experiment results are loaded yet. Run Cell 8 first, or set RUN_EXPERIMENTS = False if you only want to combine finished batches."
        )
    print("Saved canonical outputs to:", project_config.PATHS.results_dir)
    print("Saved figures to:", project_config.PATHS.figures_dir)
    if RUN_NAME is not None:
        print("This batch run was also written under:", project_config.PATHS.run_dir)

if COMBINE_AFTER_RUN or not RUN_EXPERIMENTS:
    combine_targets = COMBINE_RUN_NAMES
    if combine_targets is None:
        combine_targets = sorted(
            path.name for path in project_config.ProjectPaths(project_root).runs_dir.iterdir() if path.is_dir()
        )
    if combine_targets:
        print("Combining batch runs:", combine_targets)
        combined_results = combine_experiment_runs(root_dir=project_root, run_names=combine_targets)
        print("Rebuilt canonical outputs from batch runs.")
    else:
        print("No batch runs available to combine.")

if combined_results is None:
    combined_results = results if RUN_EXPERIMENTS else None

if combined_results is not None:
    print("Final selected models:")
    display(combined_results["final"])
    print("Summary by model:")
    display(combined_results["summary"])
    if "model_tradeoff" in combined_results:
        print("Model trade-off summary:")
        display(combined_results["model_tradeoff"])
    if "preprocessing_tradeoff" in combined_results:
        print("Preprocessing trade-off summary:")
        display(combined_results["preprocessing_tradeoff"])
    if "cv_detailed" in combined_results:
        print("Detailed CV rows available:", len(combined_results["cv_detailed"]))

if EXPORT_RUNTIME_ARTIFACTS_ARCHIVE:
    runtime_archive_path = export_runtime_artifacts_archive(
        project_root=project_root,
        artifacts_root=artifacts_root,
        run_name=RUN_NAME,
        download=DOWNLOAD_RUNTIME_ARTIFACTS_ARCHIVE,
    )
    if runtime_archive_path is not None:
        print("Archive ready for manual download:", runtime_archive_path)
elif not USE_GOOGLE_DRIVE:
    runtime_artifacts_dir = project_root / "artifacts" if artifacts_root is None else artifacts_root / "artifacts"
    print("Runtime artifacts directory:", runtime_artifacts_dir)
    print("Download or archive this directory before the Colab runtime resets.")

print("Challenge protocol reminder:")
print((project_config.PATHS.results_dir / "challenge_protocol.json").read_text(encoding="utf-8"))

## Challenge Reminder

Keep challenge accuracy out of presentation and poster material. The notebook can compute and save the private challenge results, but those numbers are only for the report and the instructor email workflow.